In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 48 samples and 9986 columns.
First few rows of the data:


,Months,Survival Status,sample_id,Tumor exome data,Normal exome data a,Tumor RNAseq data,Normal RNA seq,Race,Dx Date,Survival Date,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,63.4,0.0,5,X,NaN,X,NaN,Chinese,2004-02-23,2009-05-10,...,7.953216,6.880651,8.201768,8.172487,6.538856,7.478783,9.094757,8.576978,7.572553,8.143199
1,8.4,1.0,6,X,NaN,X,NaN,Chinese,2004-03-10,2004-11-17,...,7.694184,6.722906,7.988022,7.789258,5.651215,6.951462,8.818602,8.100091,6.968440,7.802846
2,22.7,1.0,15,X,X,X,NaN,Chinese,2005-08-29,2007-07-10,...,5.728599,4.288251,5.573461,5.697862,3.887059,4.919057,6.646728,6.397195,5.485431,5.815798
3,14.6,1.0,18,X,X,X,NaN,Chinese,2006-05-08,2007-07-19,...,5.639418,4.762175,5.660045,5.970535,4.085786,5.140975,6.639852,7.412492,7.142715,5.853071
4,23.1,0.0,20,X,X,X,NaN,Chinese,2006-10-07,2008-08-29,...,6.201917,4.753634,6.057103,6.255138,4.379792,5.403289,7.126437,6.983460,5.742906,6.198644


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for Months or Survival Status: 0
Number of samples with complete data for both variables: 48
Number of censored samples (event_var == 0.0): 25
Number of events (event_var == 1.0): 23
Censored ratio (among complete): 25 / 48 = 0.521 (52.1%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE60052/description.json
